# Séance 2 — Pandas : structures et exploration

**Durée pratique : 3 h 30** &nbsp;·&nbsp; Ateliers 2.1 à 2.3

## Ce que vous saurez faire à la fin

- charger une même source depuis trois formats et réconcilier les types obtenus ;
- sélectionner et filtrer sans déclencher de `SettingWithCopyWarning` ;
- produire automatiquement un rapport d'exploration sur un jeu de données inconnu.

Le jeu de travail est `ventes_brutes.csv` : environ 17 600 lignes de commandes,
volontairement dégradées. Vous allez le retrouver à chaque séance jusqu'à la fin du module.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [22]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


---
## Atelier 2.1 — Lire une source, vraiment (50 min)

### Partie guidée : la lecture naïve et ce qu'elle cache

In [23]:
sales = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')

print('Dimensions :', sales.shape)
sales.head()

Dimensions : (17609, 11)


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison
0,CMD0003112,2024/01/18 19:53,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,Lille
1,CMD0000107,08/01/2024,C02791,P0028,M03,2,70.48,NaN,boutique,livre,Toulouse
2,CMD0010050,2023-04-15,990031,P0017,M04,3,691.12,0.0,boutique,livre,MARSEILLE
3,CMD0001979,06 Nov 2024,C01883,P0010,M04,4,570.25,0.0,WEB,retourne,Marseille
4,CMD0010513,2024-11-02,C00571,P0024,M02,2,1058.05,10.0,web,livre,Marseille


In [24]:
sales.dtypes

id_commande            str
date_commande          str
id_client              str
id_produit             str
id_magasin             str
quantite             int64
prix_unitaire          str
remise_pct         float64
canal                  str
statut                 str
ville_livraison        str
dtype: object

Regardez `prix_unitaire`. Le type est `object`, autrement dit du texte, alors qu'il s'agit
d'un montant. Cherchons pourquoi.

In [25]:
# Isoler les valeurs qui ne se convertissent pas en nombre
as_number = pd.to_numeric(sales['prix_unitaire'], errors='coerce')
unparsable = sales.loc[as_number.isna(), 'prix_unitaire']

print('Valeurs non convertibles :', len(unparsable))
print(unparsable.head(8).tolist())

Valeurs non convertibles : 1230
['129,12 EUR', '952,06 EUR', '1062,70 EUR', '591,16 EUR', '175,45 EUR', '524,21 EUR', '882,73 EUR', '1107,70 EUR']


Une partie des prix a été saisie avec une virgule décimale et un symbole monétaire.
Une lecture qui ignore ce détail produit une colonne texte, et tout calcul en aval échoue
silencieusement ou renvoie une erreur bien plus loin dans le pipeline.

**C'est la règle à retenir : ne jamais faire confiance à l'inférence de types.** Vérifiez
systématiquement `dtypes` après chaque lecture.

In [26]:
def parse_price(series):
    """Convertit une colonne de prix mixte (nombre ou texte '123,45 EUR') en float."""
    text = series.astype(str).str.replace(' EUR', '', regex=False)
    text = text.str.replace(',', '.', regex=False).str.strip()
    return pd.to_numeric(text, errors='coerce')


unit_price = parse_price(sales['prix_unitaire'])
print('Valeurs encore non convertibles :', unit_price.isna().sum())
print(unit_price.describe().round(2))

Valeurs encore non convertibles : 0
count    17609.00
mean       616.94
std        344.75
min         60.90
25%        278.66
50%        583.33
75%        926.79
max       1287.23
Name: prix_unitaire, dtype: float64


### Partie autonome

In [27]:
# Q1. Charger l'extrait des ventes depuis les trois formats disponibles,
#     puis comparer les dtypes obtenus pour la colonne 'date_commande'.

sample_csv = pd.read_csv(RAW_DIR / 'ventes_brutes.csv')   # TODO : attention, ventes_extrait.xlsx n'est pas un CSV
sample_xlsx = pd.read_excel(RAW_DIR / 'ventes_extrait.xlsx')  # TODO : pd.read_excel
sample_json = pd.read_json(RAW_DIR / 'ventes_extrait.json')  # TODO : pd.read_json

for label, frame in [('csv', sample_csv), ('xlsx', sample_xlsx), ('json', sample_json)]:
    print(f"{label:<6} lignes={len(frame):<6} dtype date={frame['date_commande'].dtype}")

csv    lignes=17609  dtype date=str
xlsx   lignes=4000   dtype date=str
json   lignes=4000   dtype date=str


**Question à traiter par écrit dans la cellule suivante.** Les trois formats donnent-ils
le même type pour `date_commande` ? Le même nombre de lignes ? Que se passerait-il si votre
pipeline acceptait indifféremment ces trois sources ?

*Votre réponse :*

Les trois formats donnent le même type (object/texte) pour date_commande : aucun n'infère automatiquement une vraie date, contrairement à ce qu'on pourrait attendre de l'Excel. Le nombre de lignes diffère (17 609 pour le CSV ventes_brutes contre 4 000 pour les extraits xlsx/json), mais uniquement parce que ce sont des fichiers différents, pas à cause du format lui-même. Si le pipeline acceptait indifféremment ces trois sources sans normalisation explicite des types après lecture, on risquerait de propager des colonnes mal typées en aval — un datetime traité comme du texte ne se trie pas correctement et empêche tout calcul de durée. C'est pourquoi il faut systématiquement vérifier dtypes après chaque lecture, quel que soit le format d'entrée.



In [28]:
# Q2. Relire ventes_brutes.csv en une seule instruction, en imposant :
#     - id_client, id_produit, id_magasin comme chaînes de caractères ;
#     - la chaîne vide et 'NC' comme valeurs manquantes ;
#     - seulement les colonnes id_commande, date_commande, id_produit, quantite,
#       prix_unitaire, statut.
#     Indice : paramètres dtype, na_values et usecols.

EXPECTED_COLUMNS = ['id_commande', 'date_commande', 'id_produit',
                    'quantite', 'prix_unitaire', 'statut']

sales_subset = pd.read_csv(
    RAW_DIR / 'ventes_brutes.csv',
    dtype={'id_client': 'object', 'id_produit': 'object', 'id_magasin': 'object'},
    na_values=['', 'NC'],
    usecols=EXPECTED_COLUMNS,
)     # TODO

assert list(sales_subset.columns) == EXPECTED_COLUMNS
assert sales_subset['id_produit'].dtype == object
print('OK —', sales_subset.shape)

OK — (17609, 6)


---
## Atelier 2.2 — Sélection et filtrage (60 min)

### Partie guidée : `loc`, `iloc` et le piège de la copie

In [29]:
sales['prix_unitaire'] = parse_price(sales['prix_unitaire'])
sales['montant'] = sales['quantite'] * sales['prix_unitaire']

# loc travaille sur les ÉTIQUETTES (noms de colonnes, valeurs d'index)
print(sales.loc[0:2, ['id_commande', 'quantite', 'montant']])
print()
# iloc travaille sur les POSITIONS entières
print(sales.iloc[0:2, [0, 5, -1]])

  id_commande  quantite  montant
0  CMD0003112         3   481.98
1  CMD0000107         2   140.96
2  CMD0010050         3  2073.36

  id_commande  quantite  montant
0  CMD0003112         3   481.98
1  CMD0000107         2   140.96


Notez la différence sur les bornes : `loc[0:2]` renvoie **trois** lignes (borne incluse),
`iloc[0:2]` en renvoie **deux** (borne exclue, comme le slicing Python).

In [30]:
# Filtrage booléen : combiner des conditions avec & et |, chaque condition entre parenthèses
large_orders = sales[(sales['montant'] > 1000) & (sales['statut'] == 'livre')]
print('Commandes livrées de plus de 1000 EUR :', len(large_orders))

# query() est souvent plus lisible quand les conditions s'accumulent
same_result = sales.query("montant > 1000 and statut == 'livre'")
print('Même résultat :', len(same_result) == len(large_orders))

Commandes livrées de plus de 1000 EUR : 6039
Même résultat : True


In [31]:
# LE PIÈGE : modifier un sous-ensemble extrait par filtrage
cancelled = sales[sales['statut'] == 'annule']

# La ligne suivante déclenche un SettingWithCopyWarning : pandas ne sait pas si
# `cancelled` est une vue sur `sales` ou une copie indépendante.
cancelled['montant'] = 0

print('Montant dans sales pour les annulées :',
      sales.loc[sales['statut'] == 'annule', 'montant'].head(3).tolist())
print("-> la modification n'a PAS été propagée : le travail est perdu")

Montant dans sales pour les annulées : [1637.24, 1077.2, 4642.95]
-> la modification n'a PAS été propagée : le travail est perdu


**Les deux écritures correctes**, selon l'intention :

```python
# Intention A : modifier le DataFrame d'origine
sales.loc[sales['statut'] == 'annule', 'montant'] = 0

# Intention B : travailler sur une copie indépendante
cancelled = sales[sales['statut'] == 'annule'].copy()
cancelled['montant'] = 0
```

Le message d'avertissement est le symptôme d'une ambiguïté dans votre code, pas un bruit
à faire taire.

### Partie autonome

In [32]:
# Q3. Extraire les commandes qui remplissent TOUTES ces conditions :
#     - statut 'livre' ;
#     - quantite comprise entre 1 et 10 inclus ;
#     - montant strictement supérieur à la médiane des montants des commandes livrées ;
#     - canal contenant 'web', quelle que soit la casse.
#     Le résultat doit être une COPIE indépendante.

delivered_median = sales.loc[sales['statut'] == 'livre', 'montant'].median()

target_orders = sales[
    (sales['statut'] == 'livre') &
    (sales['quantite'].between(1, 10)) &
    (sales['montant'] > delivered_median) &
    (sales['canal'].str.strip().str.lower().str.contains('web'))
].copy()  # TODO

assert isinstance(target_orders, pd.DataFrame)
assert target_orders['quantite'].between(1, 10).all()
assert (target_orders['statut'] == 'livre').all()
assert target_orders['canal'].str.strip().str.lower().eq('web').all()
print('OK —', len(target_orders), 'commandes retenues')

OK — 2895 commandes retenues


In [33]:
# Q4. Sans utiliser groupby, calculer le montant total des commandes livrées
#     pour chacun des trois canaux (après normalisation de la casse).
#     Un dictionnaire {canal: total} est attendu.

canal_normalized = sales['canal'].str.strip().str.lower()
delivered = sales['statut'] == 'livre'

revenue_by_channel = {}
for channel in ['web', 'boutique', 'telephone']:
    mask = delivered & (canal_normalized == channel)
    revenue_by_channel[channel] = sales.loc[mask, 'montant'].sum()  # TODO

assert set(revenue_by_channel) == {'web', 'boutique', 'telephone'}
print('OK')
for channel, total in sorted(revenue_by_channel.items(), key=lambda item: -item[1]):
    print(f'  {channel:<12} {total:>14,.2f} EUR'.replace(',', ' '))

OK
  web           15 011 930.71 EUR
  boutique       9 861 116.79 EUR
  telephone      7 049 647.82 EUR


---
## Atelier 2.3 — Un rapport d'exploration réutilisable (100 min)

Face à un jeu de données inconnu, les mêmes questions reviennent toujours. Plutôt que de
les reposer à la main à chaque fois, vous allez écrire une fonction qui y répond.
**Cette fonction vous servira jusqu'à la fin du module, y compris sur votre projet.**

### Partie guidée : les briques

In [34]:
print('--- shape ---'); print(sales.shape)
print('\n--- info ---'); sales.info(memory_usage='deep')

--- shape ---
(17609, 12)

--- info ---
<class 'pandas.DataFrame'>
RangeIndex: 17609 entries, 0 to 17608
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id_commande      17609 non-null  str    
 1   date_commande    17259 non-null  str    
 2   id_client        17609 non-null  str    
 3   id_produit       17609 non-null  str    
 4   id_magasin       17609 non-null  str    
 5   quantite         17609 non-null  int64  
 6   prix_unitaire    17609 non-null  float64
 7   remise_pct       15668 non-null  float64
 8   canal            17609 non-null  str    
 9   statut           17609 non-null  str    
 10  ville_livraison  17609 non-null  str    
 11  montant          17609 non-null  float64
dtypes: float64(3), int64(1), str(8)
memory usage: 2.5 MB


In [35]:
print('--- taux de valeurs manquantes ---')
missing_rate = (sales.isna().mean() * 100).round(2).sort_values(ascending=False)
print(missing_rate[missing_rate > 0])

print('\n--- cardinalité des colonnes texte ---')
for column in sales.select_dtypes(include='object').columns:
    print(f'  {column:<18} {sales[column].nunique():>6} modalités')

--- taux de valeurs manquantes ---
remise_pct       11.02
date_commande     1.99
dtype: float64

--- cardinalité des colonnes texte ---
  id_commande         17369 modalités
  date_commande        6321 modalités
  id_client            3052 modalités
  id_produit             40 modalités
  id_magasin              9 modalités
  canal                   9 modalités
  statut                  3 modalités
  ville_livraison        33 modalités


In [36]:
print('--- modalités de canal, telles quelles ---')
print(sales['canal'].value_counts())

--- modalités de canal, telles quelles ---
canal
web             7013
boutique        4684
telephone       2352
WEB             1060
BOUTIQUE         724
  web            650
  boutique       525
TELEPHONE        364
  telephone      237
Name: count, dtype: int64


Neuf modalités pour ce qui devrait en compter trois. La casse et les espaces parasites
créent de faux niveaux. Un `value_counts()` brut est précisément l'outil qui révèle ce
genre de problème : c'est pourquoi il doit figurer dans le rapport automatique.

### Partie autonome : la fonction `profile_dataframe`

In [37]:
def profile_dataframe(df, name='jeu de données', max_cardinality=25):
    """Affiche un rapport d'exploration standard.

    Doit produire, dans cet ordre :
      1. le nom, les dimensions et l'empreinte mémoire ;
      2. le nombre de lignes strictement dupliquées ;
      3. un tableau par colonne : type, nombre de valeurs manquantes, taux en %,
         nombre de valeurs distinctes ;
      4. les statistiques descriptives des colonnes numériques ;
      5. pour chaque colonne texte de cardinalité inférieure à max_cardinality,
         la répartition des modalités.

    Ne renvoie rien : la fonction affiche.
    """
    print(f'--- {name} ---')
    print('dimensions :', df.shape)
    print('mémoire    :', f'{df.memory_usage(deep=True).sum() / 1024:,.1f} Ko'.replace(',', ' '))

    duplicate_count = df.duplicated().sum()
    print('doublons   :', duplicate_count)

    summary = pd.DataFrame({
        'type': df.dtypes,
        'manquants': df.isna().sum(),
        'taux_manquants_%': (df.isna().mean() * 100).round(2),
        'valeurs_distinctes': df.nunique(),
    })
    print('\n--- colonnes ---')
    print(summary)

    numeric_columns = df.select_dtypes(include='number')
    if not numeric_columns.empty:
        print('\n--- statistiques numériques ---')
        print(numeric_columns.describe().round(2))

    text_columns = df.select_dtypes(include='object').columns
    for column in text_columns:
        if df[column].nunique() < max_cardinality:
            print(f'\n--- {column} ---')
            print(df[column].value_counts())
    # TODO
    ...
profile_dataframe(sales, name='ventes_brutes')

--- ventes_brutes ---
dimensions : (17609, 12)
mémoire    : 2 579.5 Ko
doublons   : 240

--- colonnes ---
                    type  manquants  taux_manquants_%  valeurs_distinctes
id_commande          str          0              0.00               17369
date_commande        str        350              1.99                6321
id_client            str          0              0.00                3052
id_produit           str          0              0.00                  40
id_magasin           str          0              0.00                   9
quantite           int64          0              0.00                  68
prix_unitaire    float64          0              0.00               15839
remise_pct       float64       1941             11.02                   5
canal                str          0              0.00                   9
statut               str          0              0.00                   3
ville_livraison      str          0              0.00                  33
montan

In [38]:
# Q5. Appliquer la fonction aux deux autres jeux de données et vérifier qu'elle
#     se comporte correctement sur des structures différentes.

customers = pd.read_csv(RAW_DIR / 'clients.csv')
sensors = pd.read_csv(RAW_DIR / 'capteurs.csv')

profile_dataframe(customers, name='clients')
profile_dataframe(sensors, name='capteurs')

--- clients ---
dimensions : (3060, 5)
mémoire    : 223.2 Ko
doublons   : 0

--- colonnes ---
                     type  manquants  taux_manquants_%  valeurs_distinctes
id_client             str          0              0.00                3060
date_inscription      str          0              0.00                1545
segment               str          0              0.00                   3
ville                 str          0              0.00                  32
age               float64        185              6.05                  86

--- statistiques numériques ---
           age
count  2875.00
mean     46.51
std      80.88
min     -16.00
25%      31.00
50%      40.00
75%      49.00
max     999.00

--- segment ---
segment
Particulier      1542
Professionnel     997
Association       521
Name: count, dtype: int64
--- capteurs ---
dimensions : (38916, 5)
mémoire    : 2 432.4 Ko
doublons   : 300

--- colonnes ---
                  type  manquants  taux_manquants_%  valeurs_distinctes

In [39]:
# Q6. Déplacer la fonction dans src/exploration.py, puis l'importer ici.
#     Le notebook doit rester lisible : le code réutilisable vit dans src/.

import sys
sys.path.insert(0, str(ROOT / 'src'))

from exploration import profile_dataframe   # décommentez une fois le fichier créé

---
## Ce que le rapport révèle déjà

Rédigez ici, en cinq à dix lignes, la liste des anomalies que votre rapport a mises au jour
sur `ventes_brutes`. Ce texte est le point de départ de la séance 3 et le premier élément
de votre note méthodologique de projet.

*Vos observations :*

1. 240 lignes strictement dupliquées dans le jeu de données, à traiter avant toute analyse pour ne pas fausser les agrégats.
2. `quantite` contient des valeurs aberrantes : moyenne à 4.38 mais un maximum de 1199, très éloigné du reste de la distribution (75ᵉ percentile à seulement 3).
3. `montant` présente le même type d'anomalie, avec un maximum dépassant 1,1 million d'euros pour une médiane autour de 1029 — probablement lié aux valeurs aberrantes de `quantite` ou `prix_unitaire`.
4. `remise_pct` a environ 11% de valeurs manquantes (1941 lignes sur 17 609), à décider si on les impute ou si on les exclut selon l'usage prévu.
5. `canal` affiche 9 modalités au lieu des 3 attendues (`web`, `boutique`, `telephone`), à cause d'espaces parasites et d'incohérences de casse (`'Web'`, `' web '`, `'WEB'`...) , un nettoyage systématique (`.str.strip().str.lower()`) est nécessaire avant toute agrégation par canal.
6. `prix_unitaire` était initialement lue comme du texte (`object`) à cause d'un format mixte (`"123,45 EUR"` avec virgule décimale et symbole monétaire), révélant l'importance de ne jamais faire confiance à l'inférence de types de pandas.

---
## Livrable de la séance

- `src/exploration.py` contenant la fonction `profile_dataframe`, importable ;
- ce notebook exécuté, avec les six questions complétées ;
- la liste écrite des anomalies constatées.